# Predicting Customer Purchase Behaviour (E-commerce)
**Logistic Regression with Random-Forest feature importance - code companion to the project report**

| Report section | Notebook section |
|---|---|
| Dataset summary & EDA | 2 |
| Feature engineering + correlation heatmap | 3 |
| Feature selection (Random Forest importance) | 4 |
| Train/test split (80/20) | 5 |
| 5-fold cross-validation | 6 |
| Final test evaluation (metrics, confusion matrix, ROC) | 7 |
| Feature coefficients / model insight | 8 |
| Extras: model comparison, tuning, lift & cost-based threshold | 9 |
| Report vs notebook check | 10 |
| Save model & score a new customer | 11 |

**Dataset:** Kaggle - *Predict Customer Purchase Behavior* (rabieelkharoua). Upload the CSV when prompted, or let the notebook download it with `kagglehub`.
If neither works, a synthetic demo dataset is generated so the notebook still runs (its numbers will NOT match the report).

## 0. Setup

In [ ]:
import os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, roc_curve)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 1. Configuration & data loading

In [ ]:
DATA_PATH = "/content/customer_purchase_data.csv"   # change if your file has another name/location
TARGET = "PurchaseStatus"

# Optional: IQR outlier filter. The report says 'no severe outliers' yet its split totals 1,388 rows (not 1,500),
# so some rows were evidently dropped. Switch to True to experiment; the exact original rule is unknown.
DROP_OUTLIERS = False
IQR_COLS = ["AnnualIncome", "TimeSpentOnWebsite"]

In [ ]:
def make_synthetic(n=1500, seed=RANDOM_STATE):
    """Demo data with the same columns/ranges as the Kaggle set and a built-in purchase signal."""
    from scipy.optimize import brentq
    r = np.random.default_rng(seed)
    d = pd.DataFrame({
        "Age": r.integers(18, 71, n),
        "Gender": r.integers(0, 2, n),
        "AnnualIncome": r.uniform(20_000, 150_000, n),
        "NumberOfPurchases": r.integers(0, 21, n),
        "ProductCategory": r.integers(0, 5, n),
        "TimeSpentOnWebsite": r.uniform(1, 60, n),
        "LoyaltyProgram": (r.random(n) < 0.33).astype(int),
        "DiscountsAvailed": r.integers(0, 6, n),
    })
    z = lambda c: (d[c] - d[c].mean()) / d[c].std()
    s = (0.9 * z("TimeSpentOnWebsite") + 1.0 * z("DiscountsAvailed") + 2.4 * d["LoyaltyProgram"]
         - 0.9 * z("Age") + 0.8 * z("AnnualIncome") + 0.3 * z("NumberOfPurchases") + r.normal(0, 0.6, n))
    c = brentq(lambda c: (1 / (1 + np.exp(-(s - c)))).mean() - 0.432, -20, 20)
    d[TARGET] = (r.random(n) < 1 / (1 + np.exp(-(s - c)))).astype(int)
    return d

df_raw, USING_SYNTHETIC = None, False
if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
else:
    try:                                   # 1) upload dialog (Colab)
        from google.colab import files
        print("Upload customer_purchase_data.csv (or press Cancel to try kagglehub)...")
        up = files.upload()
        if up:
            df_raw = pd.read_csv(list(up.keys())[0])
    except ImportError:
        pass
    if df_raw is None:                     # 2) kagglehub download
        try:
            import kagglehub
            p = kagglehub.dataset_download("rabieelkharoua/predict-customer-purchase-behavior-dataset")
            csvs = [f for f in os.listdir(p) if f.endswith(".csv")]
            df_raw = pd.read_csv(os.path.join(p, csvs[0]))
            print("Downloaded via kagglehub:", csvs[0])
        except Exception as e:
            pass
if df_raw is None:                         # 3) synthetic fallback
    df_raw, USING_SYNTHETIC = make_synthetic(), True
    print("No data found -> using SYNTHETIC demo data (results will not match the report).")

expected = ["Age", "Gender", "AnnualIncome", "NumberOfPurchases", "ProductCategory",
            "TimeSpentOnWebsite", "LoyaltyProgram", "DiscountsAvailed", TARGET]
missing = [c for c in expected if c not in df_raw.columns]
if missing:
    raise KeyError(f"Missing columns {missing}. Available: {list(df_raw.columns)}")
df = df_raw[expected].copy()
print("Shape:", df.shape)
df.head()

## 2. Exploratory data analysis

In [ ]:
print("Shape:", df.shape)
df.info()
print("\nDuplicate rows:", df.duplicated().sum(), "| Missing values:", int(df.isna().sum().sum()))
display(df.describe().T)

prop = (df[TARGET].value_counts(normalize=True).sort_index() * 100).rename("proportion (%)")
print("\nPurchaseStatus proportion:")
display(prop.to_frame())
print(f"Loyalty-programme members: {df['LoyaltyProgram'].mean():.1%} | mean discounts availed: {df['DiscountsAvailed'].mean():.2f}")

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
sns.countplot(x=TARGET, data=df, palette=["#E63946", "#3AAF6E"], ax=ax[0, 0]); ax[0, 0].set_title("Purchase status (0 = no, 1 = yes)")
sns.histplot(df["Age"], bins=25, color="#2E86AB", ax=ax[0, 1]); ax[0, 1].set_title("Age")
sns.histplot(df["AnnualIncome"], bins=30, color="#6A4C93", ax=ax[0, 2]); ax[0, 2].set_title("Annual income")
sns.boxplot(x=TARGET, y="TimeSpentOnWebsite", data=df, palette=["#E63946", "#3AAF6E"], ax=ax[1, 0]); ax[1, 0].set_title("Time on website vs purchase")
sns.boxplot(x=TARGET, y="DiscountsAvailed", data=df, palette=["#E63946", "#3AAF6E"], ax=ax[1, 1]); ax[1, 1].set_title("Discounts availed vs purchase")
(df.groupby("LoyaltyProgram")[TARGET].mean() * 100).plot.bar(color=["#F4A261", "#3AAF6E"], ax=ax[1, 2], rot=0)
ax[1, 2].set_ylabel("% purchased"); ax[1, 2].set_title("Purchase rate by loyalty membership")
plt.tight_layout(); plt.show()

# Quick numeric check of the boxplot claims (also shows the *target-based* relationships the heatmap in the report lacks)
display(df.groupby(TARGET)[["TimeSpentOnWebsite", "DiscountsAvailed", "Age", "AnnualIncome", "LoyaltyProgram"]].mean())

In [ ]:
if DROP_OUTLIERS:
    before = len(df)
    for c in IQR_COLS:
        q1, q3 = df[c].quantile([0.25, 0.75]); iqr = q3 - q1
        df = df[df[c].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
    print(f"Outlier filter removed {before - len(df)} rows -> {len(df)} remain")
else:
    print("Outlier filter off - all rows kept.")

## 3. Feature engineering & correlation heatmap
The report uses three engineered features whose formulas are not stated. They are **inferred from the heatmap** (signs and sizes of correlations) - edit the formulas if yours differ:
- `IncomePerPurchase` = AnnualIncome / (NumberOfPurchases + 1)
- `DiscountRate` = DiscountsAvailed / (NumberOfPurchases + 1)
- `EngagementScore` = NumberOfPurchases x TimeSpentOnWebsite

In [ ]:
def add_features(d):
    d = d.copy()
    d["IncomePerPurchase"] = d["AnnualIncome"] / (d["NumberOfPurchases"] + 1)
    d["DiscountRate"]      = d["DiscountsAvailed"] / (d["NumberOfPurchases"] + 1)
    d["EngagementScore"]   = d["NumberOfPurchases"] * d["TimeSpentOnWebsite"]
    return d

df_fe = add_features(df)
heat_cols = ["Age", "AnnualIncome", "NumberOfPurchases", "TimeSpentOnWebsite", "DiscountsAvailed",
             "IncomePerPurchase", "EngagementScore", "DiscountRate"]
plt.figure(figsize=(8, 6.5))
sns.heatmap(df_fe[heat_cols].corr(), cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (Numeric Features)"); plt.tight_layout(); plt.show()

print("Correlation of each numeric feature with PurchaseStatus:")
display(df_fe[heat_cols + ["LoyaltyProgram"]].corrwith(df_fe[TARGET]).sort_values(ascending=False).to_frame("corr with target"))

## 4. Feature selection - Random Forest importance (interpretation only)

In [ ]:
feat_rf = ["Age", "Gender", "AnnualIncome", "NumberOfPurchases", "ProductCategory", "TimeSpentOnWebsite",
           "LoyaltyProgram", "DiscountsAvailed", "IncomePerPurchase", "EngagementScore", "DiscountRate"]
rf_imp = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(df_fe[feat_rf], df_fe[TARGET])
imp = pd.Series(rf_imp.feature_importances_, index=feat_rf).sort_values(ascending=False)
display(imp.to_frame("importance"))

plt.figure(figsize=(8, 5))
imp.sort_values().plot.barh(color="#4C72B0"); plt.title("Feature Importance (Random Forest)")
plt.tight_layout(); plt.show()

## 5. Pre-processing & 80/20 train-test split
Scaling and one-hot encoding live inside a `Pipeline`, so they are fitted on training data only (no leakage, including inside cross-validation).

In [ ]:
num_cols = ["Age", "AnnualIncome", "NumberOfPurchases", "TimeSpentOnWebsite", "DiscountsAvailed",
            "IncomePerPurchase", "EngagementScore", "DiscountRate"]
cat_cols = ["Gender", "ProductCategory", "LoyaltyProgram"]
all_feats = num_cols + cat_cols

X, y = df_fe[all_feats], df_fe[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"Purchase rate - train {y_train.mean():.3f} | test {y_test.mean():.3f}")

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols),
    ])

def make_model(clf=None):
    return Pipeline([("prep", make_preprocessor()), ("clf", clf if clf is not None else LogisticRegression(max_iter=1000))])

## 6. Logistic regression - 5-fold stratified cross-validation (training set)

In [ ]:
model = make_model()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
cv_means = {m: cv[f"test_{m}"].mean() for m in scoring}
for m in scoring:
    print(f"{m:9s} (mean over 5 folds): {cv_means[m]:.4f}   +/- {cv[f'test_{m}'].std():.4f}")

## 7. Final evaluation on the held-out test set

In [ ]:
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_prob),
}
for k, v in test_metrics.items():
    print(f"{k:10s}: {v:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=2))
tn, fp, fn, tp = cm.ravel()
print(f"Business reading: {fp} non-buyers wrongly targeted (wasted contacts); {fn} real buyers missed.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["No purchase", "Purchase"], yticklabels=["No purchase", "Purchase"])
ax[0].set_title("Confusion Matrix"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, label=f"ROC curve (AUC = {test_metrics['AUC']:.3f})")
ax[1].plot([0, 1], [0, 1], "--", color="#D1855C")
ax[1].set_xlabel("False Positive Rate"); ax[1].set_ylabel("True Positive Rate"); ax[1].set_title("ROC Curve"); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

gap = test_metrics["Accuracy"] - cv_means["accuracy"]
print(f"Test accuracy minus CV accuracy = {gap:+.4f}. With only {len(y_test)} test rows, a gap of a few points is within sampling noise;")
print("it supports 'no obvious overfitting' but is not proof of it.")

## 8. Feature coefficients & odds ratios (model insight)

In [ ]:
prep, clf = model.named_steps["prep"], model.named_steps["clf"]
names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
coef = pd.DataFrame({"Feature": names, "Coefficient": clf.coef_[0]})
coef["Odds Ratio"] = np.exp(coef["Coefficient"])
coef = coef.sort_values("Coefficient", ascending=False).reset_index(drop=True)
print(f"Intercept (b0): {clf.intercept_[0]:.4f}")
display(coef)

plt.figure(figsize=(8, 6))
c = coef.sort_values("Coefficient")
plt.barh(c["Feature"], c["Coefficient"], color=["#E63946" if v < 0 else "#3AAF6E" for v in c["Coefficient"]])
plt.axvline(0, color="k", lw=0.8); plt.title("Logistic regression coefficients (numeric features standardised)")
plt.tight_layout(); plt.show()
print("Note: numeric coefficients are per 1 standard deviation; binary/one-hot ones are per 1 unit, so compare sizes within each group only.")

## 9. Extras beyond the report
### 9a. Model comparison (5-fold CV, ROC-AUC) and tuning of the regularisation strength

In [ ]:
cands = {
    "Logistic Regression": make_model(),
    "Random Forest": make_model(RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": make_model(HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
}
rows = {}
for n, m in cands.items():
    s = cross_val_score(m, X_train, y_train, cv=skf, scoring="roc_auc", n_jobs=-1)
    rows[n] = {"CV AUC (mean)": s.mean(), "Std": s.std()}
display(pd.DataFrame(rows).T)

gs = GridSearchCV(make_model(), {"clf__C": [0.01, 0.1, 1, 10, 100]}, cv=skf, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
print(f"Best C = {gs.best_params_['clf__C']} | CV AUC = {gs.best_score_:.4f} | test AUC = {roc_auc_score(y_test, gs.predict_proba(X_test)[:, 1]):.4f}")

### 9b. Lift / cumulative gains - how much better than random targeting?

In [ ]:
d = pd.DataFrame({"y": y_test.values, "p": y_prob}).sort_values("p", ascending=False).reset_index(drop=True)
d["decile"] = pd.qcut(d.index, 10, labels=False) + 1
g = d.groupby("decile").agg(customers=("y", "size"), buyers=("y", "sum"))
g["response_rate"] = g["buyers"] / g["customers"]
g["lift"] = g["response_rate"] / d["y"].mean()
g["cum_capture"] = g["buyers"].cumsum() / d["y"].sum()
display(g)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].bar(g.index, g["lift"], color="#2E86AB"); ax[0].axhline(1, color="red", ls="--"); ax[0].set_title("Lift by decile"); ax[0].set_xlabel("Decile (1 = highest predicted probability)")
ax[1].plot([0] + list(g.index * 10), [0] + list(g["cum_capture"] * 100), marker="o", label="Model")
ax[1].plot([0, 100], [0, 100], "--", color="grey", label="Random"); ax[1].set_title("Cumulative gains")
ax[1].set_xlabel("% of customers contacted"); ax[1].set_ylabel("% of buyers captured"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Contacting the top 30% by score captures {g.loc[3, 'cum_capture']:.0%} of buyers in the test set.")

### 9c. Cost-based threshold (illustrative)
The 0.5 threshold is not necessarily optimal for the business. Edit the two assumptions below to match real campaign economics.

In [ ]:
CONTACT_COST = 1.0          # cost of targeting one customer (assumption)
PROFIT_PER_PURCHASE = 8.0   # profit from one conversion (assumption)

ths = np.linspace(0.05, 0.95, 91)
profit = []
for t in ths:
    pr = y_prob >= t
    profit.append((pr & (y_test.values == 1)).sum() * PROFIT_PER_PURCHASE - pr.sum() * CONTACT_COST)
best_t = ths[int(np.argmax(profit))]
at_half = ((y_prob >= 0.5) & (y_test.values == 1)).sum() * PROFIT_PER_PURCHASE - (y_prob >= 0.5).sum() * CONTACT_COST

plt.figure(figsize=(7, 4)); plt.plot(ths, profit); plt.axvline(best_t, color="green", ls="--", label=f"best = {best_t:.2f}")
plt.axvline(0.5, color="grey", ls=":", label="0.50"); plt.xlabel("Threshold"); plt.ylabel("Campaign profit (test set)")
plt.title("Expected profit vs threshold"); plt.legend(); plt.tight_layout(); plt.show()
print(f"Profit at 0.50: {at_half:,.0f} | at best threshold {best_t:.2f}: {max(profit):,.0f}")

## 10. Report vs notebook check
On the original Kaggle data the notebook should land close to the report (exact match needs the same row filtering and random split, which the report does not specify).

In [ ]:
report = {
    "CV accuracy": (0.8045, cv_means["accuracy"]), "CV precision": (0.7991, cv_means["precision"]),
    "CV recall": (0.7761, cv_means["recall"]), "CV F1": (0.7871, cv_means["f1"]), "CV ROC-AUC": (0.8876, cv_means["roc_auc"]),
    "Test accuracy": (0.8345, test_metrics["Accuracy"]), "Test precision": (0.8281, test_metrics["Precision"]),
    "Test recall": (0.8154, test_metrics["Recall"]), "Test F1": (0.8217, test_metrics["F1 Score"]), "Test AUC": (0.8924, test_metrics["AUC"]),
}
chk = pd.DataFrame(report, index=["Report", "Notebook"]).T
chk["Abs. diff"] = (chk["Report"] - chk["Notebook"]).abs()
chk["Within 0.03"] = chk["Abs. diff"] <= 0.03
display(chk)
print("Report confusion matrix: [[126, 22], [24, 106]]  |  Notebook:", cm.tolist())
if USING_SYNTHETIC:
    print("Synthetic demo data in use - differences are expected.")

## 11. Save the model & score a new customer

In [ ]:
joblib.dump(model, "purchase_model_logreg.joblib")
print("Saved purchase_model_logreg.joblib")

def predict_customer(age, gender, annual_income, n_purchases, product_category, time_on_site, loyalty, discounts):
    row = pd.DataFrame([{"Age": age, "Gender": gender, "AnnualIncome": annual_income, "NumberOfPurchases": n_purchases,
                         "ProductCategory": product_category, "TimeSpentOnWebsite": time_on_site,
                         "LoyaltyProgram": loyalty, "DiscountsAvailed": discounts}])
    p = model.predict_proba(add_features(row)[all_feats])[0, 1]
    return p

p = predict_customer(age=28, gender=1, annual_income=70_000, n_purchases=9, product_category=2,
                     time_on_site=42, loyalty=1, discounts=4)
print(f"Purchase probability: {p:.1%} -> {'target with campaign' if p >= 0.5 else 'low priority'}")